# Whale 单信号有效性实验室

**数据**：`data/cleaned/cleaned_1m_with_whale.csv`（1m + whale）。

**流程**：加载 → whale 时间窗 → 信号库 → **IS-only** sweep（`composite_is`）→ **组合图**（等权 + 波动率目标）。实现细节在 `tool/whale_signal_lab.py`，Notebook 只保留参数与调用。

**频率**：15m / 30m / 1h；**OOS**：窗口内时间轴末 30%（`OOS_TEST_FRAC`）。选信号 **不用** `test_sharpe`。组合净值图为英文。

**第 5 节（文末）**：PPT 用的大类划分、方法论流程与可复制的摘要表（大类计数、执行参数、Top20 分布）。

**硬约束**：进入排序前，样本内（IS）交易触发次数须 **> 15**（即 `is_trigger_count >= MIN_IS_TRIGGER_FOR_SWEEP`，见 `tool/whale_signal_lab.py`）。


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "tool").exists() and (ROOT.parent / "tool").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tool.cleaned_table import load_cleaned_minute_whale_csv
from tool.newmath import apply_formulas, numeric_to_float32
from tool.whale_signal_lab import (
    OOS_TEST_FRAC,
    BAR_MINUTES_LIST,
    EXEC_PROFILES,
    CFG_BASE,
    trim_whale_window,
    fill_whale_missing,
    list_whale_numeric_columns,
    resample_1m_with_whale,
    build_whale_alpha_library,
    run_whale_signal_sweep,
    plot_whale_ensemble_figures,
    build_raw_signal_matrix_1m,
    MIN_IS_TRIGGER_FOR_SWEEP,
)

plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

print("ROOT:", ROOT, "| OOS_TEST_FRAC:", OOS_TEST_FRAC)
print("BAR_MINUTES_LIST:", BAR_MINUTES_LIST, "| exec profiles:", len(EXEC_PROFILES))


In [ ]:
# --- 路径与 whale 列（可扩展 WHALE_EXTRA_NUMERIC：如 d_oi_1m）---
WHALE_EXTRA_NUMERIC: list[str] = []

CLEAN_WHALE_CSV = ROOT / "data" / "cleaned" / "cleaned_1m_with_whale.csv"
df1 = load_cleaned_minute_whale_csv(CLEAN_WHALE_CSV, root_dir=ROOT)
df1 = numeric_to_float32(df1, exclude=("time_utc",))
df1 = apply_formulas(df1, {"ret_1": "ts_returns(close, 1)"})

WHALE_BASE_COLS = list_whale_numeric_columns(df1, WHALE_EXTRA_NUMERIC)
df1 = trim_whale_window(df1, WHALE_BASE_COLS)
fill_whale_missing(df1, WHALE_BASE_COLS, policy="zero")

print("shape:", df1.shape, "| UTC:", df1["time_utc"].iloc[0], "->", df1["time_utc"].iloc[-1])


## 2. 重采样自检、信号库与 Sweep

- `resample_1m_with_whale`：1m → 15m/30m/1h 聚合 OHLCV + whale，再算 bar 上 `ret_1`（见 `tool/whale_signal_lab.py` 文档字符串）。
- `build_whale_alpha_library`：故事线（含 momentum / reverting / volatility）+ 每列 whale 的 scan；已移除 `whale_cross`。
- `run_whale_signal_sweep`：`composite_is` 排序，**不含 test**；**IS 触发次数 > 15** 硬过滤后才参与排名。


In [ ]:
# 30m 重采样自检
_r = resample_1m_with_whale(df1, 30)
print("30m resampled rows:", len(_r), "cols:", len(_r.columns))
del _r

SIGNAL_LIBRARY = build_whale_alpha_library(WHALE_BASE_COLS)
print("formula groups:", {k: len(v) for k, v in SIGNAL_LIBRARY.items()})
print("Hard filter (IS triggers): must be >=", MIN_IS_TRIGGER_FOR_SWEEP, "(strictly > 15)")

print("Running sweep …")
RESULTS = run_whale_signal_sweep(df1, SIGNAL_LIBRARY)
print("RESULTS rows (after IS trigger + obs filters):", len(RESULTS))

REF_COLS = [
    "signal", "freq", "bar_minutes", "composite_is", "train_sharpe", "robust_score",
    "wf_sharpe_min", "train_turnover", "test_sharpe", "test_turnover",
    "is_trigger_count", "os_trigger_count", "trigger_count_total",
]
REF_COLS = [c for c in REF_COLS if c in RESULTS.columns]
display(RESULTS[REF_COLS].head(30))

# 只保留 composite 最优 Top-20（composite_is 已加入低触发次数惩罚）
RESULTS_TOP20 = RESULTS.head(20).copy()
print("Top-20 rows:", len(RESULTS_TOP20))
display(RESULTS_TOP20[REF_COLS].head(20))


def _flatten_formula_library(signal_library: dict) -> dict[str, str]:
    out: dict[str, str] = {}
    for group in signal_library.values():
        for k, v in group.items():
            out[str(k)] = str(v)
    return out


def _base_name_from_signal(sig: str) -> str:
    name = sig
    if name.endswith("__inv"):
        name = name[: -len("__inv")]
    if name.startswith("sig_") and "__" in name:
        # sig_whale_flow__alert_signed_z60 -> alert_signed_z60
        name = name.split("__", 1)[1]
    return name


def _formula_for_signal(sig: str, fmap: dict[str, str]) -> str:
    base_name = _base_name_from_signal(sig)
    base_formula = fmap.get(base_name, base_name)
    if sig.endswith("__inv"):
        return f"-({base_formula})"
    return base_formula


def _economic_interpretation(sig: str, formula: str) -> str:
    text = []
    if "sig_whale_momentum" in sig:
        text.append("价格动量（多 horizon 收益）与鲸鱼资金流/仓位结合，用于趋势跟随或波动调整后的方向确认。")
    if "sig_whale_reverting" in sig:
        text.append("警报活跃度高时，结合价格在滚动区间内的相对位置（峰/谷）做反向博弈。")
    if "sig_whale_volatility" in sig:
        text.append("用鲸鱼变量短期波动或「有警报 bar 占比」刻画交投/regime，再作为交易状态输入。")
    if "ts_zscore" in formula:
        text.append("将因子标准化为相对历史分位偏离，强调异常冲击而非绝对量级。")
    if "ts_delta" in formula:
        text.append("关注边际变化而不是存量，捕捉鲸鱼行为的加速度。")
    if "ts_mean" in formula:
        text.append("先平滑后决策，降低高频噪声导致的误触发。")
    if "safe_div" in formula:
        text.append("使用比率结构做尺度归一，减少市场体量变化的干扰。")
    if "ts_rank" in formula and "close" in formula:
        text.append("用价格在窗口内的分位判断短期峰谷位置。")
    if "ts_std" in formula:
        text.append("对短期波动率水平做刻画，识别鲸鱼侧变量的不稳定期。")
    if "ifcond" in formula:
        text.append("用指示变量聚合「活跃分钟」再标准化，近似交投强度占比。")
    if "__inv" in sig or formula.strip().startswith("-("):
        text.append("该信号取反使用，表示原始行为在当前样本中更偏向反向交易解释。")
    if not text:
        text.append("该信号直接由鲸鱼行为原始量构建，反映资金流与仓位变化对后续价格的影响。")
    return " ".join(text)


_formula_map = _flatten_formula_library(SIGNAL_LIBRARY)
out_md = ROOT / "main" / "Whale_Top20_Signal_Logic.md"

lines = [
    "# Whale Single-Signal Top 20 (by composite_is)",
    "",
    "- 排序口径：`composite_is`（IS-only），并加入低触发次数惩罚项。",
    "- 硬约束：样本内 `is_trigger_count` 必须 **> 15**（代码中为 `>= MIN_IS_TRIGGER_FOR_SWEEP`）才进入排序；下表均已满足。",
    "- 触发次数定义：`abs(position)` 从 0/近0 进入非0 的事件数（多空合并统计）。",
    "- 信号库含：flow / position_book / attention / **momentum / reverting / volatility** / scan（已移除 cross）。",
    "- 下表信号均来自当前 notebook 运行结果 Top-20。",
    "",
]

for rank, (_, row) in enumerate(RESULTS_TOP20.iterrows(), start=1):
    sig = str(row["signal"])
    formula = _formula_for_signal(sig, _formula_map)
    interp = _economic_interpretation(sig, formula)
    lines += [
        f"## {rank}. `{sig}`",
        "",
        f"- 频率: `{row['freq']}` (bar={int(row['bar_minutes'])}m)",
        f"- 公式: `{formula}`",
        f"- 评分: composite_is={float(row['composite_is']):.4f}",
        f"- 触发次数: IS={int(row.get('is_trigger_count', 0))}, OS={int(row.get('os_trigger_count', 0))}, Total={int(row.get('trigger_count_total', 0))}",
        f"- 经济含义: {interp}",
        "",
    ]

out_md.write_text("\n".join(lines), encoding="utf-8")
print("Saved:", out_md)


## 3. Ensemble（Top-5/10/20/30 × 等权、波动率目标 10%/20%/40%）

两张图：**Equal weight**；**Vol target**（三子图，对应三个年化波动目标）。竖线 = `train_test_masks_whale_last_pct(df1)` 下第一个 OOS bar（IS | OOS）。

下一小节 **3b**：Top30 原始信号在 **1m 对齐**后的相关矩阵。


In [ ]:
import warnings
warnings.filterwarnings("ignore")
plot_whale_ensemble_figures(df1, RESULTS, SIGNAL_LIBRARY, CFG_BASE)


## 3b. Top30 信号相关性（统一对齐到 1m）

对 `RESULTS` 前 30 行：每条按其 `bar_minutes` 在对应频率上计算 **raw 公式信号**（未经过 `signal_to_position`），再用与组合仓位相同的规则 **forward-fill 到 1m 时间轴**，最后算 **Pearson** 相关矩阵。

若关心 **执行后仓位** 之间的相关，可把 `build_raw_signal_matrix_1m` 换成 `build_position_matrix_1m(..., CFG_BASE)` 再 `.corr()`。


In [ ]:
# Top30：原始信号对齐 1m → 相关矩阵 + 热力图
TOP_N_CORR = 30
if RESULTS is not None and not RESULTS.empty:
    top30_corr = RESULTS.head(min(TOP_N_CORR, len(RESULTS))).copy()
    SIG_MAT_1M = build_raw_signal_matrix_1m(top30_corr, df1, SIGNAL_LIBRARY)
    CORR_TOP30 = SIG_MAT_1M.corr()
    display(CORR_TOP30.round(3))

    n = len(CORR_TOP30.columns)
    fig, ax = plt.subplots(figsize=(max(12, n * 0.35), max(10, n * 0.32)))
    im = ax.imshow(CORR_TOP30.values, cmap="coolwarm", vmin=-1.0, vmax=1.0, aspect="auto")
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(CORR_TOP30.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(CORR_TOP30.columns, fontsize=7)
    ax.set_title("Top-30 raw signal correlation (1m-aligned, Pearson)")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    plt.tight_layout()
    plt.show()
else:
    print("RESULTS empty; run sweep first.")
